# Day 1: Text processing (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — split, slice, unpack, set, dict.

In [ ]:
text = "the quick brown fox jumps over the lazy dog the end"
words = text.split()

In [ ]:
assert words[1:4] == ["quick", "brown", "fox"]

In [ ]:
first, *middle, last = words

In [ ]:
assert (first, last) == ("the", "end") and len(middle) == len(words) - 2
assert len(set(words)) == 9

In [ ]:
counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1

In [ ]:
assert counts["the"] == 3

**Then, vary it:**

- VARIATION: build counts with dict.setdefault instead of .get
- VARIATION: get the last 3 words with a negative slice; reverse them with [::-1]

## `s2_flow.py`: Flow — comprehensions, match, generator expression.

In [ ]:
words = "the quick brown fox jumps over the lazy dog".split()

In [ ]:
assert [len(w) for w in words] == [3, 5, 5, 3, 5, 4, 3, 4, 3]
assert {w for w in words if len(w) > 4} == {"quick", "brown", "jumps"}
assert {w: len(w) for w in words}["quick"] == 5

In [ ]:
def classify(w):
    match len(w):
        case n if n <= 3: return "short"
        case 4 | 5: return "medium"
        case _: return "long"

In [ ]:
assert [classify(w) for w in ("fox", "quick", "elephants")] == ["short", "medium", "long"]

In [ ]:
gen = (w.upper() for w in words)

In [ ]:
assert next(gen) == "THE"

**Then, vary it:**

- VARIATION: add a "huge" class for len > 8
- VARIATION: turn the dict comprehension into one that keeps only the longest word per length

## `s3_functions.py`: Functions — kw-only args, closure, decorator.

In [ ]:
import functools
def normalize(text, *, lower=True, strip_punct=True):
    if lower: text = text.lower()
    if strip_punct: text = "".join(c for c in text if c.isalnum() or c.isspace())
    return text

In [ ]:
assert normalize("Hello, World!") == "hello world"
assert normalize("Hello!", lower=False) == "Hello"

In [ ]:
def make_min_len(n):
    return lambda w: len(w) >= n

In [ ]:
assert make_min_len(4)("four") and not make_min_len(4)("it")

In [ ]:
def logged(fn):
    calls = []
    @functools.wraps(fn)
    def wrapper(*a, **k):
        calls.append((a, k))
        return fn(*a, **k)
    wrapper.calls = calls
    return wrapper
@logged
def shout(s): return s.upper()
shout("a"); shout("b")


In [ ]:
assert len(shout.calls) == 2 and shout.__name__ == "shout"

**Then, vary it:**

- VARIATION: make `logged` take a prefix argument (decorator factory)
- VARIATION: remove @functools.wraps and see what shout.__name__ becomes

## `s4_classes.py`: Classes — dataclass, ordering, dunder protocol.

In [ ]:
from dataclasses import dataclass
@dataclass(order=True, frozen=True)
class WordStat:
    count: int
    word: str
a, b = WordStat(3, "the"), WordStat(1, "fox")

In [ ]:
assert b < a and sorted([a, b])[0] == b
assert repr(a) == "WordStat(count=3, word='the')"

In [ ]:
class Bag:
    def __init__(self, words): self._w = list(words)
    def __len__(self): return len(self._w)
    def __iter__(self): return iter(self._w)
    def __contains__(self, w): return w in self._w
c = Bag(["a", "b"])

In [ ]:
assert len(c) == 2 and "a" in c and list(c) == ["a", "b"]

**Then, vary it:**

- VARIATION: add __getitem__ so c[0] works, and __add__ to merge two Bags
- VARIATION: try mutating a frozen WordStat and catch the error

## `s5_errors.py`: Errors — custom exception, pathlib file IO, context manager.

In [ ]:
import contextlib, tempfile
from pathlib import Path
class EmptyTextError(ValueError): pass
def first_word(text):
    if not text.strip(): raise EmptyTextError("no words")
    return text.split()[0]

In [ ]:
try:
    first_word("   ")
except EmptyTextError as e:
    assert str(e) == "no words"
else:
    raise AssertionError

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "a.txt"
    p.write_text("hello\nworld\n")
    with p.open() as f:
        assert [l.rstrip() for l in f] == ["hello", "world"]
@contextlib.contextmanager
def tag(log, name):
    log.append(f"<{name}>")
    try: yield
    finally: log.append(f"</{name}>")
log = []
with tag(log, "b"): log.append("hi")

In [ ]:
assert log == ["<b>", "hi", "</b>"]

**Then, vary it:**

- VARIATION: raise inside the `with tag` block; confirm the closing tag still appears
- VARIATION: write a class-based context manager (__enter__/__exit__) doing the same

## `s6_stdlib.py`: Stdlib — Counter, defaultdict, re, itertools, json.

In [ ]:
import re, json, itertools
from collections import Counter, defaultdict
text = "the cat and the hat and the bat"
c = Counter(text.split())

In [ ]:
assert c.most_common(1) == [("the", 3)]

In [ ]:
assert re.findall(r"\b\w{3}\b", "a cat sat") == ["cat", "sat"]
groups = defaultdict(list)
for w in text.split(): groups[len(w)].append(w)

In [ ]:
assert groups[3][:2] == ["the", "cat"]

In [ ]:
assert list(itertools.pairwise("abc")) == [("a", "b"), ("b", "c")]

In [ ]:
import re, json, itertools

assert json.loads(json.dumps(dict(c)))["and"] == 2

**Then, vary it:**

- VARIATION: find all words ending in "at" with one regex
- VARIATION: use Counter arithmetic: c - Counter(["the"]) and c + c

## `s7_concurrency.py`: Concurrency — thread pool map.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
docs = ["a b c", "d e", "f"]
def count(d): return len(d.split())
with ThreadPoolExecutor(max_workers=3) as ex:
    results = list(ex.map(count, docs))

In [ ]:
assert results == [3, 2, 1] and sum(results) == 6

**Then, vary it:**

- VARIATION: use ex.submit + as_completed; why does order change?
- VARIATION: explain why threads won't speed up CPU-heavy count() (GIL)

## `s8_quality.py`: Quality — type hints, doctest, timeit.

In [ ]:
import doctest, timeit
def word_count(text: str) -> dict[str, int]:
    """
    >>> word_count("a b a")
    {'a': 2, 'b': 1}
    """
    out: dict[str, int] = {}
    for w in text.split(): out[w] = out.get(w, 0) + 1
    return out
assert doctest.testmod().failed == 0
t = timeit.timeit(lambda: word_count("a b a " * 100), number=100)

In [ ]:
assert t < 5

**Then, vary it:**

- VARIATION: add a second doctest for the empty string
- VARIATION: rewrite with Counter and compare timeit numbers

## `kata.py`: KATA — top-N words (plain loops on purpose; Day 7 rebuilds this idiomatically).

In [ ]:
def top_words(text, n=3):
    counts = {}
    for raw in text.lower().split():
        w = "".join(c for c in raw if c.isalnum())
        if w: counts[w] = counts.get(w, 0) + 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:n]

In [ ]:
assert top_words("The cat. The hat! A cat?", 2) == [("cat", 2), ("the", 2)]
assert top_words("") == []

**Then, vary it:**

- KATA: delete the function body and rewrite from memory; add a stop-words param